# Chapter 9 - Lab 2: <font color='blue'>Fraud Investigation via Adversarial Debate</font>

**<font color='purple'>Goal</font>**:
When the Lab 1 fraud screener flags a claim, you need a deeper look. This lab implements the investigation step using an **adversarial debate** pattern:

* A **Fraud Advocate** argues for fraud.
* A **Legitimacy Advocate** argues against fraud, providing innocent explanations for every red flag.
* A senior **Investigation Decision** agent weighs both arguments and produces a determination.

Forcing a model to argue *both* sides surfaces evidence quality and reasoning weaknesses that a single-pass classifier would miss. Each advocate must also acknowledge *the weaknesses in its own case* — a discipline that anchors the final decision in something defensible.

**<font color='purple'>Tech stack</font>**:

* **LlamaIndex** — `FunctionAgent` with explicit handoffs.
* **OpenAI** `gpt-4o`.
* **Pydantic** — typed `FraudArgument` schema from `common.py`.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## 3. Bootstrap the shared models and helpers

Chapter 9 labs share a `common.py` module that defines the Pydantic models and helpers used across the chapter. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%209/common.py',
        'common.py',
    )

from common import FraudArgument, SUSPICIOUS_CLAIM_SUMMARY

print('Common helpers loaded.')
print('Suspicious claim under investigation:\n' + SUSPICIOUS_CLAIM_SUMMARY)

## 4. Define the three debate agents

The two advocate agents have no tools — they reason from the claim summary. Their system prompts force them to (a) build the strongest possible case for their side and (b) acknowledge their own weaknesses. The investigation decision agent then weighs both.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model='gpt-4o', temperature=0)

fraud_advocate = FunctionAgent(
    name='FraudAdvocate',
    description='Argue the case for fraud.',
    system_prompt=(
        'You are an insurance fraud investigator. Review the claim and build the '
        'strongest possible case that it is fraudulent. Cite specific evidence. '
        'Also identify the weaknesses in your own argument — where is your case thin? '
        'Be thorough but honest. Hand off to InvestigationDecision.'
    ),
    llm=llm, tools=[], can_handoff_to=['InvestigationDecision'],
)

legitimacy_advocate = FunctionAgent(
    name='LegitimacyAdvocate',
    description='Argue the claim is legitimate.',
    system_prompt=(
        'You are a consumer-protection advocate. For every red flag in the claim, '
        'provide an innocent alternative explanation. Build the strongest possible '
        'case that the claim is legitimate, and acknowledge your own weaknesses. '
        'Your role is to ensure no legitimate claim is wrongly denied. '
        'Hand off to InvestigationDecision.'
    ),
    llm=llm, tools=[], can_handoff_to=['InvestigationDecision'],
)

investigation_decision = FunctionAgent(
    name='InvestigationDecision',
    description='Weigh fraud and legitimacy arguments.',
    system_prompt=(
        'You are the senior fraud investigator. You have arguments from a Fraud '
        'Advocate and a Legitimacy Advocate. Evaluate (1) evidence quality, '
        '(2) reasoning strength, (3) acknowledged weaknesses on each side. '
        'Produce one of: confirmed_fraud / likely_fraud / inconclusive / likely_legitimate. '
        'Include a confidence (0-1) and a recommended action.'
    ),
    llm=llm, tools=[], can_handoff_to=[],
)

## 5. Wire the debate as a workflow

The Fraud Advocate goes first. It hands off to InvestigationDecision — but the workflow keeps the LegitimacyAdvocate in the pool, so the decision agent can prompt it for its side of the argument. (In a real implementation you would model this as a parallel pattern with an explicit synchronisation step; this simplified single-handler version keeps the lab focused.)

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow

workflow = AgentWorkflow(
    agents=[fraud_advocate, legitimacy_advocate, investigation_decision],
    root_agent='FraudAdvocate',
    initial_state={'claim_summary': SUSPICIOUS_CLAIM_SUMMARY},
)

## 6. Run the debate

We send a structured prompt that asks the Fraud Advocate to begin, then explicitly instructs the workflow that the Legitimacy Advocate should also weigh in before InvestigationDecision concludes.

In [ ]:
user_msg = (
    'Investigate the following potentially fraudulent claim. '
    'FraudAdvocate: argue for fraud. Then LegitimacyAdvocate must respond with '
    'innocent explanations. Finally, InvestigationDecision weighs both sides.\n\n'
    f'Claim summary:\n{SUSPICIOUS_CLAIM_SUMMARY}'
)

handler = workflow.run(user_msg=user_msg)

async for event in handler.stream_events():
    if hasattr(event, 'current_agent_name'):
        print(f'\n=== {event.current_agent_name} ===')
    if hasattr(event, 'delta') and event.delta:
        print(event.delta, end='', flush=True)

final = await handler
print('\n\nFinal determination:\n' + str(final))

## 7. Structured arguments (optional, advanced)

In production you would want each advocate to produce a **typed** `FraudArgument` instead of free-form text. The schema from `common.py` is shown below — you can extend the agents with `output_type=FraudArgument` (when using PydanticAI) to enforce this structure.

In [ ]:
import json

sample_fraud_argument = FraudArgument(
    position='fraud',
    confidence=0.85,
    evidence=[
        'Claim filed 5 days after policy inception',
        'Three prior claims in 14 months',
        'Claim amount = 94% of policy limit',
        'No witnesses, no security footage',
    ],
    reasoning='Pattern matches classic burn-and-claim auto fraud.',
    weaknesses=[
        'Police report does not affirmatively allege fraud.',
        'Prior claims may all be legitimate.',
    ],
)
print(json.dumps(sample_fraud_argument.model_dump(), indent=2))

## 8. Results

You forced a model to argue both sides of a fraud question, then judged the debate. **What to notice about the adversarial-debate pattern:**

* **Quality > single-pass classification.** Forcing both arguments surfaces evidence quality and   reasoning weaknesses that a one-shot 'is this fraud?' prompt would miss.
* **Self-acknowledged weaknesses anchor the decision.** When both sides must name their own   weaknesses, the judge has something concrete to weigh.
* **This is structured deliberation, not a chat.** The agents do not free-form chat — they   produce arguments with typed structure. Easier to audit, easier to review later.
* **Where this fits in the broader pipeline.** Lab 1's FraudScreener flags suspects with a fast   heuristic score. When the score crosses a threshold, *this* debate runs as a second-line   investigation. Cheap first, deep second.